# 04 · The same trade-off at a real UK site: Dogger Bank

Notebook 03 used a Danish site and 2 MW turbines. Here the problem moves to Dogger Bank in the
UK North Sea, with the wind climate taken from the Global Wind Atlas and 10 MW turbines
(DTU 10 MW reference turbine, 178.3 m rotor, 119 m hub height).

The wake model changes to Niayifar & Porté-Agel (2016), which lets the wake recovery depend on
turbulence intensity (set to 6 %, a typical offshore value). The lease area is 3 km × 2 km and the
minimum spacing is again 4 D (713 m).

In [ ]:
%pip install -q py_wake==2.6.20 topfarm==2.6.2 pymoo==0.6.2

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
from scipy.sparse.csgraph import minimum_spanning_tree

from py_wake.site import XRSite
from py_wake.site.xrsite import GlobalWindAtlasSite
from py_wake.examples.data.dtu10mw import DTU10MW
from py_wake.literature.gaussian_models import Niayifar_PorteAgel_2016
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize

FIG = Path("figures"); FIG.mkdir(exist_ok=True)

In [ ]:
DATA = Path("data/dogger_bank_gwa.nc")

def load_dogger_bank_site():
    """Dogger Bank wind climate from the Global Wind Atlas (54.75 N, 1.90 E, 119 m, open sea).
    Downloaded once and cached in data/ so later runs don't depend on the GWA API."""
    if DATA.exists():
        return XRSite(xr.load_dataset(DATA))
    print("Downloading Dogger Bank wind climate from the Global Wind Atlas...")
    site = GlobalWindAtlasSite(lat=54.75, long=1.90, height=119, roughness=0.0002)
    DATA.parent.mkdir(exist_ok=True)
    site.ds.to_netcdf(DATA)
    return site

In [ ]:
site_uk = load_dogger_bank_site()
wt = DTU10MW()
TI = 0.06
wf_model = Niayifar_PorteAgel_2016(site_uk, wt)

LEASE_X, LEASE_Y = 3000, 2000
MIN_SPACING = 4 * wt.diameter()     # 713 m
POP_SIZE, N_GEN, SEED = 40, 50, 1

x_init = np.linspace(0, 2400, 4).repeat(2)    # 800 m between columns
y_init = np.tile(np.linspace(0, 1600, 2), 4)  # 1600 m between rows
n_wt = len(x_init)

aep_init = wf_model(x_init, y_init, TI=TI).aep().sum().values
print(f"Initial 4x2 grid AEP at Dogger Bank: {aep_init:.1f} GWh")

In [ ]:
plt.figure(figsize=(6, 6))
site_uk.plot_wd_distribution(n_wd=12)
plt.title("Dogger Bank wind rose (Global Wind Atlas)", pad=20)
plt.savefig(FIG / "04_wind_rose_dogger_bank.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
def mst_edges(x, y):
    return minimum_spanning_tree(squareform(pdist(np.column_stack([x, y]))))

def cable_length_km(x, y):
    return mst_edges(x, y).sum() / 1000


class DoggerBankLayout(ElementwiseProblem):
    def __init__(self):
        super().__init__(n_var=2 * n_wt, n_obj=2, n_ieq_constr=1,
                         xl=[0] * (2 * n_wt),
                         xu=[LEASE_X] * n_wt + [LEASE_Y] * n_wt)

    def _evaluate(self, v, out, *args, **kwargs):
        x, y = v[:n_wt], v[n_wt:]
        aep = wf_model(x, y, TI=TI).aep().sum().values
        out["F"] = [-aep, cable_length_km(x, y)]
        out["G"] = [MIN_SPACING - pdist(np.column_stack([x, y])).min()]


def initial_population(problem, seed_row):
    # Random layouts, with the starting grid as the first member so the search
    # always has one feasible, sensible layout to build from.
    rng = np.random.default_rng(SEED)
    X0 = rng.uniform(problem.xl, problem.xu, size=(POP_SIZE, problem.n_var))
    X0[0] = seed_row
    return X0


problem = DoggerBankLayout()
X0 = initial_population(problem, np.concatenate([x_init, y_init]))
result = minimize(problem, NSGA2(pop_size=POP_SIZE, sampling=X0), ("n_gen", N_GEN), seed=SEED, verbose=False)

cable_init = cable_length_km(x_init, y_init)
pareto = pd.DataFrame({"AEP_GWh": -result.F[:, 0], "Cable_km": result.F[:, 1]}) \
    .assign(GWh_per_km_cable=lambda d: d.AEP_GWh / d.Cable_km) \
    .sort_values("AEP_GWh").reset_index(drop=True)
print(pareto.round(2).to_string())
print(f"\nInitial 4x2 grid: {aep_init:.1f} GWh, {cable_init:.2f} km")

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(pareto.Cable_km, pareto.AEP_GWh, label="Pareto front (NSGA-II)")
plt.scatter(cable_init, aep_init, color="red", marker="x", s=120, label="Initial 4x2 grid")
plt.xlabel("Inter-array cable length, MST (km)")
plt.ylabel("AEP (GWh)")
plt.title("Energy vs cable length, 8 × DTU 10 MW at Dogger Bank")
plt.legend(); plt.grid(True, alpha=0.4)
plt.savefig(FIG / "04_pareto_front_dogger_bank.png", dpi=150, bbox_inches="tight")
plt.show()

## Highest-AEP layout vs the starting grid

A genetic algorithm doesn't track individual turbines (turbine 3 in one layout has no relation to
turbine 3 in another), so the two layouts are overlaid rather than linked with arrows.
The orange lines are the MST cable network of the optimised layout.

In [ ]:
best = np.argmin(result.F[:, 0])
x_best, y_best = result.X[best, :n_wt], result.X[best, n_wt:]

plt.figure(figsize=(12, 7))
plt.plot([0, LEASE_X, LEASE_X, 0, 0], [0, 0, LEASE_Y, LEASE_Y, 0], "k--", alpha=0.7, label="Lease area")
for i, j in zip(*mst_edges(x_best, y_best).nonzero()):
    plt.plot([x_best[i], x_best[j]], [y_best[i], y_best[j]], color="darkorange", lw=2, zorder=1)
plt.scatter(x_init, y_init, color="darkred", marker="x", s=120, label="Initial 4x2 grid")
plt.scatter(x_best, y_best, color="forestgreen", s=150, alpha=0.8, zorder=2,
            label=f"Highest-AEP layout ({-result.F[best, 0]:.1f} GWh, {result.F[best, 1]:.2f} km)")
plt.title("Highest-AEP layout at Dogger Bank")
plt.xlabel("x (m)"); plt.ylabel("y (m)")
plt.xlim(-200, LEASE_X + 200); plt.ylim(-200, LEASE_Y + 200)
plt.legend(loc="upper right"); plt.grid(True, linestyle=":", alpha=0.6)
plt.savefig(FIG / "04_best_layout_dogger_bank.png", dpi=150, bbox_inches="tight")
plt.show()